# 07 — Hybrid physics + ML (approach C) and engineered targets

Can machine learning add anything on top of the physics model? (`src/vfm.py`, section C)

| Method | Idea |
|---|---|
| B + WC correction | `WC = trend(t) + b·(tubing_gradient − mean)`: heavier column ↔ more water. One `b` for all wells. |
| C1 residual | ML learns the relative physics error `true/physics − 1`, then `rate = physics·(1 + correction)` |
| C2 physics-as-feature | the physics prediction is one more input to the ML model (original targets) |
| C3 engineered targets | ML predicts liquid, WC, GOR → `oil = liquid·(1−WC)`, `water = liquid·WC`, `gas = oil·GOR` |

**Out-of-fold (OOF) vs in-sample:** to learn where physics is wrong, ML needs the physics errors on the training
tests. If physics is fitted on a test and then predicts that same test, its error looks too small. OOF = each
training test predicted with itself hidden (a leave-one-out inside the leave-one-out). Both versions are tested.

⏱ This notebook takes ~5 minutes (the OOF experiments refit the physics ~7,000 times).

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..', 'src').resolve()))
import load
import validation as V
import vfm

tt = pd.read_parquet(load.PROCESSED / 'test_features.parquet')

RESIDUAL_FEATURES = ['tubing_gradient', 'gas_lift_rate_mscfd', 'choke_flow_term', 'choke_p_ratio',
                     'dh_temperature_degf', 'temp_below_res', 'lift_dp', 'days']
PF = vfm.PHYSICS_FEATURES

experiments = {
    'B physics (reference)': vfm.physics,
    'B + WC tubing correction': vfm.make_physics_method(wc_correction='tubing'),
    'B + WC tubing+lift correction': vfm.make_physics_method(wc_correction='tubing+lift'),
    'C1 residual Ridge OOF': vfm.make_residual_method(RESIDUAL_FEATURES, vfm.ridge, True),
    'C1 residual Ridge in-sample': vfm.make_residual_method(RESIDUAL_FEATURES, vfm.ridge, False),
    'C1 residual LightGBM OOF': vfm.make_residual_method(RESIDUAL_FEATURES, vfm.default_lgbm, True),
    'C2 physics-as-feature LightGBM OOF': vfm.make_physics_feature_method(PF, vfm.default_lgbm, True),
    'C2 physics-as-feature LightGBM in-sample': vfm.make_physics_feature_method(PF, vfm.default_lgbm, False),
    'C2 physics-as-feature Ridge OOF': vfm.make_physics_feature_method(PF, vfm.ridge, True),
    'C3 eng. targets LightGBM no label feats': vfm.make_engineered_target_method(PF, vfm.default_lgbm, False),
    'C3 eng. targets LightGBM + label feats OOF': vfm.make_engineered_target_method(PF, vfm.default_lgbm, True, True),
    'C3 eng. targets Ridge + label feats OOF': vfm.make_engineered_target_method(PF, vfm.ridge, True, True),
}

results, predictions = [], {}
for name, method in experiments.items():
    predictions[name], _ = V.run_experiment(name, tt, method, results)

V.save_results(results, load.ROOT / 'outputs' / 'ablation_results.csv')
pd.DataFrame(results)[['experiment', 'oil_MAPE', 'water_MAPE', 'gas_MAPE', 'mean_MAPE']].round(2)

## Are the small improvements real? Paired test against physics
For each test: difference in |% error|. Wilcoxon p < 0.05 → probably real; p > 0.05 → could be luck.

In [ ]:
base = predictions['B physics (reference)']
for name in ['B + WC tubing correction', 'C3 eng. targets Ridge + label feats OOF']:
    print(name)
    print(V.paired_test(tt, base, predictions[name]).round(3), '\n')

**Conclusions**
- **C1 Ridge residual = physics exactly**: Ridge's internal CV picked the strongest penalty → correction ≈ 0.
  The linear model found *nothing systematic* left in the physics errors.
- **LightGBM hybrids are worse** (overfit 84 rows), even with the physics given as a feature.
- OOF vs in-sample made little difference here (physics has only 2–3 parameters per well, so in-sample
  errors are already close to real ones).
- Small water gains (WC tubing correction, C3 Ridge) are **not statistically significant**; C3 Ridge also
  makes gas worse.
- → **The physics model stays the main model.** Remaining error ≈ test measurement noise.